## 0001 MODEL COVID

### Objetivos:
-> AVALIAR MODELO LOGISTICO COM BASE NOS DADOS DA RAW TABLE



In [1]:
# pip install shap


In [2]:
import pyarrow.parquet as pq

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# cálculo statisticos
from scipy.stats import shapiro
from scipy.stats import f_oneway
import scipy.stats as stats

import statsmodels.api as sm

# modelagem
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## avaliacao do modelo
from sklearn.metrics import classification_report, roc_auc_score

## validacao cruzada
from sklearn.model_selection import cross_val_score


In [3]:
# desativando warmings
import warnings
warnings.filterwarnings("ignore")

In [4]:
#  Caminho do arquivo Parquet
file_path = "df_casos_modelo_1.parquet"

# Carregar o arquivo Parquet
df_abt = pd.read_parquet(file_path)

In [5]:
# Lista das colunas categóricas
columns_categoric = [
     # target
    'EVOLUCAO_Catego',
    
    # Demais categoricas
    'CS_SEXO', 'SG_UF_NOT', 'PUERPERA_Catego', 'ESTRANG_Catego', 'CS_RACA_Catego', 'CS_ESCOL_N_Catego', 
    'CS_ZONA_Catego', 'FATOR_RISC_Catego', 'RECEBEU_VACINA_COVID_Catego', 'VACINA_Catego', 'MAE_VAC_Catego', 
]


In [6]:
# Verificar se as colunas categóricas existem no DataFrame
missing_columns = [col for col in columns_categoric if col not in df_abt.columns]
if missing_columns:
    print(f"As seguintes colunas estão ausentes: {missing_columns}")
else:
    # Converter colunas categóricas para tipo categórico
    for col in columns_categoric:
        df_abt[col] = df_abt[col].astype('category')

    # Converter as demais colunas para tipo numérico
    numeric_columns = [col for col in df_abt.columns if col not in columns_categoric]
    for col in numeric_columns:
        df_abt[col] = pd.to_numeric(df_abt[col], errors='coerce')
    
    print("Conversão concluída com sucesso!")


Conversão concluída com sucesso!


In [7]:
df_abt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 941689 entries, 0 to 941688
Data columns (total 70 columns):
 #   Column                            Non-Null Count   Dtype   
---  ------                            --------------   -----   
 0   EVOLUCAO_Catego                   941689 non-null  category
 1   CS_SEXO                           941689 non-null  category
 2   SG_UF_NOT                         941689 non-null  category
 3   PUERPERA_Catego                   941689 non-null  category
 4   ESTRANG_Catego                    941689 non-null  category
 5   CS_RACA_Catego                    941689 non-null  category
 6   CS_ESCOL_N_Catego                 941689 non-null  category
 7   CS_ZONA_Catego                    941689 non-null  category
 8   FATOR_RISC_Catego                 687305 non-null  category
 9   RECEBEU_VACINA_COVID_Catego       941689 non-null  category
 10  VACINA_Catego                     941689 non-null  category
 11  MAE_VAC_Catego                    94168

In [8]:
# Agrupando e contando as ocorrências
df_grouped = df_abt.groupby('EVOLUCAO_Catego').size().reset_index(name='count')

# Calculando a porcentagem
df_grouped['percentage'] = (df_grouped['count'] / df_grouped['count'].sum()) * 100

# Ordenando os resultados por 'percentage' (opcional)
df_grouped = df_grouped.sort_values(by='percentage', ascending=False)

df_grouped

,EVOLUCAO_Catego,count,percentage
0,Cura,684686,72.708293
1,Óbito,257003,27.291707


In [9]:
# Verificar se as colunas categóricas existem no DataFrame
missing_columns = [col for col in columns_categoric if col not in df_abt.columns]
if missing_columns:
    print(f"As seguintes colunas estão ausentes: {missing_columns}")
else:
    # Realizar a dummificação
    df_abt_dummified = pd.get_dummies(df_abt, columns=columns_categoric, drop_first=False)
    print("Dummificação concluída com sucesso!")

    # Identificar colunas dummificadas
    dummified_columns = [col for col in df_abt_dummified.columns if any(cat in col for cat in columns_categoric)]
    print(f"Colunas que foram dummificadas: {dummified_columns}")


# df_abt_dummified.columns


Dummificação concluída com sucesso!
Colunas que foram dummificadas: ['EVOLUCAO_Catego_Cura', 'EVOLUCAO_Catego_Óbito', 'CS_SEXO_F', 'CS_SEXO_M', 'SG_UF_NOT_AC', 'SG_UF_NOT_AL', 'SG_UF_NOT_AM', 'SG_UF_NOT_AP', 'SG_UF_NOT_BA', 'SG_UF_NOT_CE', 'SG_UF_NOT_DF', 'SG_UF_NOT_ES', 'SG_UF_NOT_GO', 'SG_UF_NOT_MA', 'SG_UF_NOT_MG', 'SG_UF_NOT_MS', 'SG_UF_NOT_MT', 'SG_UF_NOT_PA', 'SG_UF_NOT_PB', 'SG_UF_NOT_PE', 'SG_UF_NOT_PI', 'SG_UF_NOT_PR', 'SG_UF_NOT_RJ', 'SG_UF_NOT_RN', 'SG_UF_NOT_RO', 'SG_UF_NOT_RR', 'SG_UF_NOT_RS', 'SG_UF_NOT_SC', 'SG_UF_NOT_SE', 'SG_UF_NOT_SP', 'SG_UF_NOT_TO', 'PUERPERA_Catego_Nao', 'PUERPERA_Catego_Sim', 'ESTRANG_Catego_Nao', 'ESTRANG_Catego_Sim', 'CS_RACA_Catego_Amarela', 'CS_RACA_Catego_Branca', 'CS_RACA_Catego_Indigena', 'CS_RACA_Catego_Parda', 'CS_RACA_Catego_Preta', 'CS_ESCOL_N_Catego_Fundamental 1 ciclo 1 a 5 serie', 'CS_ESCOL_N_Catego_Fundamental 2 ciclo 6 a 9 serie', 'CS_ESCOL_N_Catego_Ignorado', 'CS_ESCOL_N_Catego_Medio 1 ao 3 ano', 'CS_ESCOL_N_Catego_Nao se aplica',

In [10]:
# Separando features e target
X = df_abt_dummified[
    [
        'CS_SEXO_F', 
        
        'SG_UF_NOT_AC', 'SG_UF_NOT_AM', 'SG_UF_NOT_AP',  #'SG_UF_NOT_AL', 
        'SG_UF_NOT_BA', 'SG_UF_NOT_CE', 'SG_UF_NOT_DF', 'SG_UF_NOT_ES', 
        'SG_UF_NOT_GO', 'SG_UF_NOT_MA', 'SG_UF_NOT_MG', 'SG_UF_NOT_MS', 
        'SG_UF_NOT_MT', 'SG_UF_NOT_PA', 'SG_UF_NOT_PB', 'SG_UF_NOT_PE', 
        'SG_UF_NOT_PI', 'SG_UF_NOT_PR', 'SG_UF_NOT_RJ', 'SG_UF_NOT_RN', 
        'SG_UF_NOT_RO', 'SG_UF_NOT_RR', 'SG_UF_NOT_RS', 'SG_UF_NOT_SC', 
        'SG_UF_NOT_SE', 'SG_UF_NOT_SP', 'SG_UF_NOT_TO', 
        
        'PUERPERA_Catego_Sim', 'ESTRANG_Catego_Sim', 

        'CS_RACA_Catego_Amarela', 'CS_RACA_Catego_Branca', 'CS_RACA_Catego_Indigena', 'CS_RACA_Catego_Parda', 'CS_RACA_Catego_Preta', 
        
        'CS_ESCOL_N_Catego_Fundamental 1 ciclo 1 a 5 serie', 'CS_ESCOL_N_Catego_Fundamental 2 ciclo 6 a 9 serie', 'CS_ESCOL_N_Catego_Ignorado', 
        'CS_ESCOL_N_Catego_Medio 1 ao 3 ano', 'CS_ESCOL_N_Catego_Nao se aplica', 'CS_ESCOL_N_Catego_Sem escolaridade Analfabeto', 
        'CS_ESCOL_N_Catego_Superior', 
        
        'CS_ZONA_Catego_Periurbana', 'CS_ZONA_Catego_Rural', 'CS_ZONA_Catego_Urbana', 
        
        'FATOR_RISC_Catego_Sim', 
        'RECEBEU_VACINA_COVID_Catego_Sim', 'VACINA_Catego_Sim', 
        'MAE_VAC_Catego_Sim',

        'NU_IDADE_N', 'DT_PERMANENCIA_UTI', 'DT_SINTOMA_INTERNACAO',
        'POS_PCROUT', 'DT_EVOLUCAO_VACICOVID_2DOSE', 
        'Penta', 'PERD_PALA', 'DT_VACICOVID_DELTA_REFDOSE', 'PCR_FLUASU', 'TripliceBacteDTP1ref', 
        'TripliceViralD1', 'Poliomielite4anos', 'Pneumococica', 'MeningococoC1ref', 'TP_FLU_PCR', 
        'PCR_SARS2', 'TripliceViralD2', 'DT_VACICOVID_REFDOSE_INTERNACAO', 'HepatiteA', 'DTPREF4e6anos', 
        'Poliomielite1ref', 'DT_DELTA_SINTOMA_VACGRIPE', 'Pneumococica1ref', 'RES_IGM', 'BCG', 
        'DT_DELTA_EVOLUCAO_VACGRIPE', 'DT_EVOLUCAO_VACICOVID_REFDOSE', 'DT_VACICOVID_1DOSE_UTI', 
         'DT_EVOLUCAO_ANTIGENICO', 'DT_VACICOVID_1DOSE_INTERNACAO',  #'DT_DELTA_SINTOMA_VACCRIANCA',
        'RES_IGA', 'DT_VACICOVID_2REFDOSE_INTERNACAO', 'DT_VACICOVID_2DOSE_UTI', 'DTP', 
        'DT_EVOLUCAO_VACICOVID_2REFDOSE', 'FebreAmarela', 'POS_AN_OUT', 'MeningococoC', 'AN_SARS2', 
        'RotavirusHumano', 'PERD_OLFT', 'Varicela', 'HepatiteBidadeMenor30dias', 'Poliomielite', 
        'DT_DELTA_EVOLUCAO_VACCRIANCA', 'DT_VACICOVID_2DOSE_INTERNACAO', 'DT_EVOLUCAO_DELTA_2REFDOSE', 
        'HepatiteB', 'DT_VACICOVID_DELTA_2DOSE', 'RES_IGG', 'DT_EVOLUCAO_VACICOVID_1DOSE', 
        
        'cluster_pni', 'distance_to_centroid_pni', 'cluster_idh',
        'Prob_obito_med', 'Prob_Contam_med'      
    ]
]

y = df_abt_dummified['EVOLUCAO_Catego_Óbito']

# Substituir valores infinitos por NaN
X.replace([np.inf, -np.inf], 0, inplace=True)



In [11]:
# Dividindo dados
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [12]:
# Padronizando os dados
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [13]:
X_train.columns

Index(['CS_SEXO_F', 'SG_UF_NOT_AC', 'SG_UF_NOT_AM', 'SG_UF_NOT_AP',
       'SG_UF_NOT_BA', 'SG_UF_NOT_CE', 'SG_UF_NOT_DF', 'SG_UF_NOT_ES',
       'SG_UF_NOT_GO', 'SG_UF_NOT_MA',
       ...
       'DT_EVOLUCAO_DELTA_2REFDOSE', 'HepatiteB', 'DT_VACICOVID_DELTA_2DOSE',
       'RES_IGG', 'DT_EVOLUCAO_VACICOVID_1DOSE', 'cluster_pni',
       'distance_to_centroid_pni', 'cluster_idh', 'Prob_obito_med',
       'Prob_Contam_med'],
      dtype='object', length=104)

In [14]:
X_test.columns

Index(['CS_SEXO_F', 'SG_UF_NOT_AC', 'SG_UF_NOT_AM', 'SG_UF_NOT_AP',
       'SG_UF_NOT_BA', 'SG_UF_NOT_CE', 'SG_UF_NOT_DF', 'SG_UF_NOT_ES',
       'SG_UF_NOT_GO', 'SG_UF_NOT_MA',
       ...
       'DT_EVOLUCAO_DELTA_2REFDOSE', 'HepatiteB', 'DT_VACICOVID_DELTA_2DOSE',
       'RES_IGG', 'DT_EVOLUCAO_VACICOVID_1DOSE', 'cluster_pni',
       'distance_to_centroid_pni', 'cluster_idh', 'Prob_obito_med',
       'Prob_Contam_med'],
      dtype='object', length=104)

In [15]:
# Resetando os índices de X_train e y_train
X_train_sm = X_train.reset_index(drop=True)
y_train_sm = y_train.reset_index(drop=True)

# Adicionando constante (intercepto) ao modelo
X_train_sm = sm.add_constant(X_train_sm)

In [16]:
y_train_sm = y_train_sm.astype(int)


In [17]:
# Adicionando constante (intercepto) ao modelo
# y_train_sm = sm.add_constant(y_train_sm)

In [18]:
print(X_train_sm.shape)  # Deve ser (659182, n_features)
print(y_train_sm.shape)  # Deve ser (659182,)


(659182, 105)
(659182,)


In [19]:
print(y_train_sm.ndim)  # Deve ser 1


1


In [20]:
constant_columns = [col for col in X_train_sm.columns if X_train_sm[col].nunique() == 1]
print("Colunas constantes:", constant_columns)
# X_train_sm = X_train_sm.drop(constant_columns, axis=1)


Colunas constantes: ['const']


In [21]:
# duplicates = X_train_sm.T[X_train_sm.T.duplicated(keep=False)]
# print(duplicates)


In [22]:
# Treinando o modelo com statsmodels
model = sm.Logit(y_train_sm, X_train_sm)
result = model.fit(disp=False)

In [23]:
# Resumo do modelo, incluindo a estatística Z de Wald
print(result.summary())

                             Logit Regression Results                            
Dep. Variable:     EVOLUCAO_Catego_Óbito   No. Observations:               659182
Model:                             Logit   Df Residuals:                   659083
Method:                              MLE   Df Model:                           98
Date:                   Thu, 09 Jan 2025   Pseudo R-squ.:                  0.1910
Time:                           17:19:55   Log-Likelihood:            -3.1285e+05
converged:                         False   LL-Null:                   -3.8669e+05
Covariance Type:               nonrobust   LLR p-value:                     0.000
                                                        coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------
const                                                -1.3212      0.004   -354.713      0.000      -1.329   

In [24]:
X_test_sm = sm.add_constant(X_test)

In [25]:
# Previsões no conjunto de teste
y_pred_proba = result.predict(X_test_sm)
y_pred = (y_pred_proba > 0.5).astype(int)

In [26]:
# Relatório de classificação
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.79      0.92      0.85    205733
        True       0.62      0.36      0.45     76774

    accuracy                           0.77    282507
   macro avg       0.71      0.64      0.65    282507
weighted avg       0.75      0.77      0.74    282507



In [27]:
# Calculando o ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC: {roc_auc}")

ROC-AUC: 0.791473061274744


In [28]:
weights = {0: 1, 1: 5}  # Atribuir maior peso para a classe minoritária

In [40]:
# Treinando o modelo
from sklearn.linear_model import LogisticRegression

# Ajustando o modelo com class_weight='balanced'
model = LogisticRegression(
    class_weight='balanced', 
    random_state=42, 
    penalty='elasticnet', 
    solver='saga', 
    l1_ratio=0.5  # 50% L1 e 50% L2
)

model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', l1_ratio=0.5, penalty='elasticnet',
                   random_state=42, solver='saga')

### Avaliação do Modelo <br>
-> Calcule métricas como:
- Acurácia: Proporção de previsões corretas.
- ROC-AUC: Avalia a capacidade do modelo de distinguir entre classes.
- Precisão e Recall: Avaliam o desempenho em cada classe.

In [41]:
# Previsões
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Relatório de classificação
print(classification_report(y_test, y_pred))

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC: {roc_auc}")

              precision    recall  f1-score   support

       False       0.88      0.70      0.78    205733
        True       0.48      0.74      0.58     76774

    accuracy                           0.71    282507
   macro avg       0.68      0.72      0.68    282507
weighted avg       0.77      0.71      0.72    282507

ROC-AUC: 0.791682203562259


1. Teste de Qui-quadrado

In [42]:
from sklearn.metrics import log_loss

# Log-verossimilhança do modelo ajustado
log_likelihood_full = -log_loss(y_test, y_pred_proba, normalize=False)

# Log-verossimilhança do modelo nulo (apenas a média de y_train)
p_null = np.mean(y_train)
log_likelihood_null = -log_loss(y_test, [p_null] * len(y_test), normalize=False)

# Estatística do qui-quadrado
chi2_stat = 2 * (log_likelihood_full - log_likelihood_null)
print(f"Estatística Qui-quadrado: {chi2_stat}")

# Valor-p
from scipy.stats import chi2
df = X_train.shape[1]  # Graus de liberdade
p_value = chi2.sf(chi2_stat, df)
print(f"Valor-p do teste Qui-quadrado: {p_value}")


Estatística Qui-quadrado: 17308.616814479174
Valor-p do teste Qui-quadrado: 0.0


In [43]:
len(X_train.columns)

104

2. Estatística Z de Wald

In [44]:
# Cálculo da estatística Z de Wald
coefficients = model.coef_[0]
standard_errors = np.sqrt(np.diag(np.linalg.inv(np.dot(X_train.T, X_train))))
z_scores = coefficients / standard_errors

# P-valores
from scipy.stats import norm
p_values = 2 * (1 - norm.cdf(np.abs(z_scores)))

# Resultado
wald_results = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': coefficients,
    'Standard_Error': standard_errors,
    'Z_Score': z_scores,
    'P_Value': p_values
})

# Exibir resultados
print(wald_results)

# Exibindo as variáveis significativas
# print(wald_df.sort_values('P-Valor').head(10))

# Filtrar variáveis com p-valor > 0,05
non_significant_vars = wald_results[wald_results['P_Value'] < 0.05]

# # Exibir variáveis com p-valor > 0,05 e suas informações completas
# print("Variáveis com p-valor > 0,05 (não significativas):")
# print(non_significant_vars.sort_values('P-Valor', ascending=True))


LinAlgError: Singular matrix

In [ ]:
non_significant_vars

3. Procedimento Stepwise

In [45]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Substituir valores infinitos por 0 no DataFrame original
X.replace([np.inf, -np.inf], 0, inplace=True)

# Divisão dos dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Reiniciar índices para evitar desalinhamento
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Padronização das variáveis
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# Adicionando a constante para o intercepto
X_train_sm = sm.add_constant(X_train)

# Ajustando o modelo inicial de regressão logística
initial_model = sm.Logit(y_train, X_train_sm).fit()
print("Resumo do Modelo Inicial:")
print(initial_model.summary())


         Current function value: 0.474608
         Iterations: 35
Resumo do Modelo Inicial:
                             Logit Regression Results                            
Dep. Variable:     EVOLUCAO_Catego_Óbito   No. Observations:               659182
Model:                             Logit   Df Residuals:                   659083
Method:                              MLE   Df Model:                           98
Date:                   Thu, 09 Jan 2025   Pseudo R-squ.:                  0.1910
Time:                           17:38:08   Log-Likelihood:            -3.1285e+05
converged:                         False   LL-Null:                   -3.8669e+05
Covariance Type:               nonrobust   LLR p-value:                     0.000
                                                        coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------
const           

In [ ]:
# Histórico para o procedimento stepwise
stepwise_history = []
aic_values = []

# Salvando o modelo inicial
stepwise_history.append(initial_model.params.index.tolist())
aic_values.append(initial_model.aic)

# Procedimento manual de stepwise com base no AIC
current_model = initial_model
while True:
    # Testar a remoção de cada variável individualmente
    aic_candidates = []
    candidate_models = []

    for col in X_train_sm.columns[1:]:  # Exclui a constante
        reduced_X = X_train_sm.drop(columns=[col])
        try:
            reduced_model = sm.Logit(y_train, reduced_X).fit(disp=False)
            aic_candidates.append((col, reduced_model.aic))
            candidate_models.append(reduced_model)
        except Exception as e:
            continue

    # Verificar se há melhoria no AIC
    if not aic_candidates:
        break
    best_candidate = min(aic_candidates, key=lambda x: x[1])

    # Se o AIC melhora, atualiza o modelo e remove a variável
    if best_candidate[1] < current_model.aic:
        print(f"Removendo variável: {best_candidate[0]} (AIC: {best_candidate[1]})")
        current_model = candidate_models[aic_candidates.index(best_candidate)]
        X_train_sm = X_train_sm.drop(columns=[best_candidate[0]])
        stepwise_history.append(current_model.params.index.tolist())
        aic_values.append(best_candidate[1])
    else:
        break

# Resumo do modelo final
print("Resumo do Modelo Final:")
print(current_model.summary())

In [31]:
# Heatmap: Histórico da Seleção de Variáveis
import seaborn as sns
import matplotlib.pyplot as plt

# Transformar histórico em DataFrame binário (1 = incluída, 0 = removida)
all_features = list(set([var for step in stepwise_history for var in step]))
heatmap_data = pd.DataFrame(index=all_features, columns=[f"Passo {i+1}" for i in range(len(stepwise_history))])

heatmap_data = heatmap_data.apply(pd.to_numeric, errors='coerce')

heatmap_data = heatmap_data.fillna(0)  # Substitui NaN por 0, por exemplo

for i, step_vars in enumerate(stepwise_history):
    heatmap_data.iloc[:, i] = heatmap_data.index.isin(step_vars).astype(int)

In [ ]:
# Comparação do Modelo Inicial vs. Final
metrics_comparison = pd.DataFrame({
    'Métrica': ['AIC', 'BIC', 'Log-Likelihood'],
    'Modelo Inicial': [initial_model.aic, initial_model.bic, initial_model.llf],
    'Modelo Final': [current_model.aic, current_model.bic, current_model.llf]
})

# Gráfico de comparação
metrics_comparison.set_index('Métrica').plot(kind='bar', figsize=(8, 6), color=['blue', 'green'])
plt.title('Comparação de Métricas entre Modelo Inicial e Final')
plt.ylabel('Valores')
plt.grid(axis='y')
plt.legend(loc='best')
plt.show()

### Interpretação dos Resultados <br>
Coeficientes positivos indicam aumento na probabilidade de óbito, enquanto coeficientes negativos indicam redução.

In [ ]:
coef = np.exp(model.coef_)
print(f"Odds ratio: {coef}")

### Validação

In [ ]:
scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
print(f"Mean ROC-AUC: {scores.mean()}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, classification_report

# Previsões
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Relatório de classificação
print(classification_report(y_test, y_pred))

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC: {roc_auc}")

# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.grid()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)

# Plotagem com seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis_r', cbar=False)
plt.title("Matriz de Confusão")
plt.xlabel("Predito")
plt.ylabel("Verdadeiro")
plt.show()

In [ ]:
# Convertendo para porcentagem
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

# Plotagem com seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(cm_percent, annot=True, fmt='.2f', cmap='viridis_r', cbar=True)
plt.title("Matriz de Confusão (%)")
plt.xlabel("Predito")
plt.ylabel("Verdadeiro")
plt.show()

In [ ]:
print(X_train.dtypes)
print(X_test.dtypes)


In [ ]:
pip install shap

In [ ]:
import shap

X_train = X_train.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')


# Criar o explainer
explainer = shap.Explainer(model, X_train)

# Calcular os valores SHAP
shap_values = explainer(X_test)

# Visualizar a importância global das features
shap.summary_plot(shap_values, X_test)

# Visualizar explicações locais
# shap.force_plot(explainer.expected_value[0], shap_values[0].values, X_test.iloc[0])

# Dependência de uma feature específica
# shap.dependence_plot('NU_IDADE_N', shap_values.values, X_test)
